#  Technical Operations Simulation
## Security Company


- **CRM** → `raw_clients.csv` — client accounts, contract types, monthly fees
- **Field Ops system** → `raw_installations.csv` — installation jobs, engineers, systems installed
- **Callout log** → `raw_callouts.csv` — reactive maintenance calls, fault types, SLA response

1. Load and audit each dataset
2. Clean and standardise
3. Consolidate into a master table using SQL
4. Engineer KPIs for the reporting framework
5. Export clean data ready for Power BI

---

In [1]:
import pandas as pd
import numpy as np
import sqlite3

## Audit and clean client data CSV

In [2]:
clients = pd.read_csv("raw_data/raw_clients.csv")

print("=== Client Data ===")
print(clients.shape)
clients.head()

=== Client Data ===
(30, 8)


,client_id,client_name,site_type,region,account_manager,contract_start,contract_type,monthly_fee_gbp
0,clt001,Tesco Express Hackney,Transport Hub,East London,Dave Okafor,2021-10-09,Maintenance,NaN
1,CLT002,tesco express hackney,Warehouse,Kent,Dave Okafor,24/11/2022,Monitoring Only,150.0
2,CLT003,Barclays Bank Stratford,Retail,East London,Dave Okafor,12/08/2021,Maintenance,120.0
3,CLT004,Premier Inn Bethnal Green,Transport Hub,Kent,Priya shah,2022-03-06,Maintenance,1000.0
4,CLT005,DHL Warehouse Dagenham,Office,East London,dave okafor,09/03/2022,Full Service,350.0


#### Check for null values and fill them with the median 

In [3]:
print(clients.isnull().sum())

client_id          0
client_name        0
site_type          0
region             0
account_manager    0
contract_start     0
contract_type      0
monthly_fee_gbp    4
dtype: int64


In [4]:
median_fee = clients['monthly_fee_gbp'].median()
clients['monthly_fee_gbp'] = clients['monthly_fee_gbp'].fillna(median_fee)
print(clients.isnull().sum()) 

client_id          0
client_name        0
site_type          0
region             0
account_manager    0
contract_start     0
contract_type      0
monthly_fee_gbp    0
dtype: int64


In [5]:
# Spot IDs that don't match the expected format
clients[clients['client_id'] != clients['client_id'].str.upper()]

,client_id,client_name,site_type,region,account_manager,contract_start,contract_type,monthly_fee_gbp
0,clt001,Tesco Express Hackney,Transport Hub,East London,Dave Okafor,2021-10-09,Maintenance,250.0
6,clt007,Lidl Walthamstow,Office,South London,Priya shah,2021-09-28,Monitoring Only,1000.0
12,clt013,Poundland Ilford,Hospitality,West London,Priya Shah,2022-10-18,Monitoring Only,250.0
18,clt019,Argos Barking,Transport Hub,North London,Lee Hutchins,2022-09-03,Monitoring Only,250.0
24,clt025,KFC Stratford,Retail,East London,Priya Shah,2021-11-11,Maintenance,120.0


In [6]:
clients['client_id'] = clients['client_id'].str.upper() #fix all ID names

In [7]:
clients['account_manager'] = clients['account_manager'].str.strip().str.title()

In [8]:
dupes = clients[clients['client_name'].str.lower().duplicated(keep=False)]
dupes

,client_id,client_name,site_type,region,account_manager,contract_start,contract_type,monthly_fee_gbp
0,CLT001,Tesco Express Hackney,Transport Hub,East London,Dave Okafor,2021-10-09,Maintenance,250.0
1,CLT002,tesco express hackney,Warehouse,Kent,Dave Okafor,24/11/2022,Monitoring Only,150.0


### Handling Duplicate: Tesco Express Hackney

Two entries found for the same client (CLT001 and CLT002). 
Confirmed with account manager that CLT001 is the correct record.
CLT002 dropped — incorrect region (Kent vs East London) and lower contract value.

In [9]:
clients = clients.drop(index=1)

In [10]:
print(clients.shape)
clients[clients['client_name'].str.lower().str.contains('tesco')]

(29, 8)


,client_id,client_name,site_type,region,account_manager,contract_start,contract_type,monthly_fee_gbp
0,CLT001,Tesco Express Hackney,Transport Hub,East London,Dave Okafor,2021-10-09,Maintenance,250.0


In [11]:
print(clients['contract_start'].tolist())

['2021-10-09', '12/08/2021', '2022-03-06', '09/03/2022', '05/04/2021', '2021-09-28', '28/10/2021', '09/11/2022', '2022-01-25', '15/01/2022', '16/09/2022', '2022-10-18', '02/02/2021', '19/11/2021', '2021-09-29', '21/08/2022', '04/04/2021', '2022-09-03', '26/06/2022', '19/10/2022', '2021-01-04', '17/10/2022', '28/06/2022', '2021-11-11', '13/05/2022', '19/06/2021', '2021-11-16', '04/05/2021', '22/07/2022']


### Fix: Mixed Date Formats in contract_start

Audit revealed two date formats present: DD/MM/YYYY and YYYY-MM-DD.
Wrote a custom parser to handle both and standardised to datetime64.

In [12]:
def parse_date(d):
    for fmt in ('%d/%m/%Y', '%Y-%m-%d'):
        try:
            return pd.to_datetime(d, format=fmt)
        except:
            continue
    return pd.NaT

clients['contract_start'] = clients['contract_start'].apply(parse_date)

In [13]:
print(clients['contract_start'].head(10).tolist())
print(clients['contract_start'].dtype)

[Timestamp('2021-10-09 00:00:00'), Timestamp('2021-08-12 00:00:00'), Timestamp('2022-03-06 00:00:00'), Timestamp('2022-03-09 00:00:00'), Timestamp('2021-04-05 00:00:00'), Timestamp('2021-09-28 00:00:00'), Timestamp('2021-10-28 00:00:00'), Timestamp('2022-11-09 00:00:00'), Timestamp('2022-01-25 00:00:00'), Timestamp('2022-01-15 00:00:00')]
datetime64[us]


In [14]:
print(clients.shape)
print(clients.isnull().sum())
print(clients.dtypes)
clients.head()

(29, 8)
client_id          0
client_name        0
site_type          0
region             0
account_manager    0
contract_start     0
contract_type      0
monthly_fee_gbp    0
dtype: int64
client_id                     str
client_name                   str
site_type                     str
region                        str
account_manager               str
contract_start     datetime64[us]
contract_type                 str
monthly_fee_gbp           float64
dtype: object


,client_id,client_name,site_type,region,account_manager,contract_start,contract_type,monthly_fee_gbp
0,CLT001,Tesco Express Hackney,Transport Hub,East London,Dave Okafor,2021-10-09,Maintenance,250.0
2,CLT003,Barclays Bank Stratford,Retail,East London,Dave Okafor,2021-08-12,Maintenance,120.0
3,CLT004,Premier Inn Bethnal Green,Transport Hub,Kent,Priya Shah,2022-03-06,Maintenance,1000.0
4,CLT005,DHL Warehouse Dagenham,Office,East London,Dave Okafor,2022-03-09,Full Service,350.0
5,CLT006,NHS Whipps Cross,Warehouse,South London,Dave Okafor,2021-04-05,Full Service,150.0


## Audit and clean installation data CSV

In [15]:
installs = pd.read_csv("raw_data/raw_installations.csv")
print(installs.shape)
installs.head(5)

(80, 10)


,job_id,client_ref,system_type,engineer,job_date,duration_hrs,install_cost_gbp,status,cameras_installed,warranty_months
0,JOB0001,CLT003,Intruder Alarm,Mike Tanner,2023-01-31,7.4,3000.0,Completed,NaN,24.0
1,JOB0002,CLT009,Intruder Alarm,Connor Mills,2023-09-06,2.4,NaN,Completed,13.0,24.0
2,JOB0003,CLT004,Intercom,mike tanner,2023-02-19,3.8,4500.0,In Progress,NaN,12.0
3,JOB0004,CLT022,CCTV,mike tanner,2023-02-20,6.2,1200.0,Completed,NaN,24.0
4,JOB0005,CLT018,Intruder Alarm,mike tanner,2023-08-18,2.7,NaN,Completed,14.0,NaN


In [16]:
print(installs.isnull().sum())

job_id                0
client_ref            0
system_type           0
engineer              0
job_date              0
duration_hrs          0
install_cost_gbp      7
status                0
cameras_installed    53
warranty_months      14
dtype: int64


In [17]:
median_install_cost = installs['install_cost_gbp'].median()

installs['install_cost_gbp'] = installs['install_cost_gbp'].fillna(median_install_cost)

In [18]:
print(installs['engineer'].value_counts())

engineer
Raj Patel       17
Mike Tanner     16
Connor Mills    14
mike tanner     14
Yemi Adeyemi    10
Sharon Webb      9
Name: count, dtype: int64


In [19]:
installs['engineer'] = installs['engineer'].str.strip().str.title()

In [20]:
print(installs['engineer'].value_counts())

engineer
Mike Tanner     30
Raj Patel       17
Connor Mills    14
Yemi Adeyemi    10
Sharon Webb      9
Name: count, dtype: int64


In [21]:
installs['job_date'] = installs['job_date'].apply(parse_date)

In [22]:
print(installs['status'].value_counts())

status
Completed           43
Failed - Revisit    13
In Progress         12
Pending             12
Name: count, dtype: int64


In [23]:
print(installs['cameras_installed'].value_counts(dropna=False))

cameras_installed
NaN     53
13.0     5
8.0      4
15.0     3
11.0     2
2.0      2
1.0      2
14.0     1
0.0      1
9.0      1
4.0      1
10.0     1
16.0     1
5.0      1
3.0      1
7.0      1
Name: count, dtype: int64


In [24]:
installs[installs['cameras_installed'].isna()]['system_type'].value_counts()

system_type
CCTV + Alarm      11
Intercom          10
Fire Alarm        10
Intruder Alarm     9
CCTV               9
Access Control     4
Name: count, dtype: int64

In [25]:
installs['cameras_installed'] = installs['cameras_installed'].fillna(0).astype(int)

In [26]:
print(installs['warranty_months'].value_counts(dropna=False))

warranty_months
24.0    35
36.0    19
NaN     14
12.0    12
Name: count, dtype: int64


In [27]:
waranty_median = installs['warranty_months'].median()
installs['warranty_months'] = installs['warranty_months'].fillna(waranty_median)

In [28]:
print(installs.shape)
print(installs.isnull().sum())
print(installs.dtypes)
installs.head()

(80, 10)
job_id               0
client_ref           0
system_type          0
engineer             0
job_date             0
duration_hrs         0
install_cost_gbp     0
status               0
cameras_installed    0
warranty_months      0
dtype: int64
job_id                          str
client_ref                      str
system_type                     str
engineer                        str
job_date             datetime64[us]
duration_hrs                float64
install_cost_gbp            float64
status                          str
cameras_installed             int64
warranty_months             float64
dtype: object


,job_id,client_ref,system_type,engineer,job_date,duration_hrs,install_cost_gbp,status,cameras_installed,warranty_months
0,JOB0001,CLT003,Intruder Alarm,Mike Tanner,2023-01-31,7.4,3000.0,Completed,0,24.0
1,JOB0002,CLT009,Intruder Alarm,Connor Mills,2023-09-06,2.4,2000.0,Completed,13,24.0
2,JOB0003,CLT004,Intercom,Mike Tanner,2023-02-19,3.8,4500.0,In Progress,0,12.0
3,JOB0004,CLT022,CCTV,Mike Tanner,2023-02-20,6.2,1200.0,Completed,0,24.0
4,JOB0005,CLT018,Intruder Alarm,Mike Tanner,2023-08-18,2.7,2000.0,Completed,14,24.0


## Audit and clean callout data CSV

In [29]:
callouts = pd.read_csv('raw_data/raw_callouts.csv')
print(callouts.shape)
callouts.head()

(150, 10)


,callout_id,client_id,fault_type,priority,raised_at,responded_at,resolved_at,sla_hrs,engineer,repeat_fault
0,CAL00001,CLT009,Camera Offline,Medium,2023-04-07 02:00,2023-04-08 10:20,2023-04-08 13:45,24,Yemi Adeyemi,False
1,CAL00002,CLT019,Cable Damage,Critical,2023-04-11 05:00,2023-04-11 08:08,2023-04-11 10:11,4,Connor Mills,False
2,CAL00003,CLT021,Power Issue,Low,2023-10-26 15:00,2023-10-27 22:50,2023-10-28 00:46,48,Raj Patel,False
3,CAL00004,CLT014,Camera Offline,High,2023-09-25 15:00,2023-09-25 22:35,2023-09-26 00:29,8,Sharon Webb,False
4,CAL00005,CLT011,Sensor Fault,Low,2023-04-30 12:00,2023-05-02 08:21,NaN,48,Connor Mills,False


In [30]:
print(callouts.isnull().sum())
print(callouts['priority'].value_counts())
print(callouts['engineer'].value_counts())

callout_id       0
client_id        0
fault_type       0
priority         0
raised_at        0
responded_at     0
resolved_at     13
sla_hrs          0
engineer         0
repeat_fault     0
dtype: int64
priority
High        47
Critical    37
Medium      35
Low         31
Name: count, dtype: int64
engineer
Connor Mills    30
Sharon Webb     27
mike tanner     27
Mike Tanner     25
Raj Patel       24
Yemi Adeyemi    17
Name: count, dtype: int64


In [31]:
callouts['engineer'] = callouts['engineer'].str.strip().str.title()

In [32]:
print(callouts['engineer'].value_counts())

engineer
Mike Tanner     52
Connor Mills    30
Sharon Webb     27
Raj Patel       24
Yemi Adeyemi    17
Name: count, dtype: int64


In [50]:
callouts['raised_at'] = pd.to_datetime(callouts['raised_at'])
callouts['responded_at'] = pd.to_datetime(callouts['responded_at'])

callouts['response_hrs'] = (
    (callouts['responded_at'] - callouts['raised_at']).dt.total_seconds() / 3600
).round(2)

callouts['sla_breached'] = callouts['response_hrs'] > callouts['sla_hrs']

## SQL database consolidation

In [77]:
conn = sqlite3.connect(':memory:')

clients.to_sql('clients', conn, index=False, if_exists='replace')
installs.to_sql('installations', conn, index=False, if_exists='replace')
callouts.to_sql('callouts', conn, index=False, if_exists='replace')

print('Tables loaded.')

Tables loaded.


In [78]:
result = pd.read_sql('''
    SELECT c.client_name, c.region, i.system_type, i.install_cost_gbp, co.fault_type, co.priority
    FROM clients c
    JOIN installations i ON i.client_ref = c.client_id
    JOIN callouts co ON co.client_id = c.client_id
    LIMIT 10
''', conn)
result

,client_name,region,system_type,install_cost_gbp,fault_type,priority
0,Tesco Express Hackney,East London,CCTV,800.0,Alarm False Trigger,High
1,Tesco Express Hackney,East London,CCTV,800.0,Cable Damage,Medium
2,Tesco Express Hackney,East London,CCTV,800.0,Door Access Failure,High
3,Tesco Express Hackney,East London,Fire Alarm,1200.0,Alarm False Trigger,High
4,Tesco Express Hackney,East London,Fire Alarm,1200.0,Cable Damage,Medium
5,Tesco Express Hackney,East London,Fire Alarm,1200.0,Door Access Failure,High
6,Tesco Express Hackney,East London,Intruder Alarm,1500.0,Alarm False Trigger,High
7,Tesco Express Hackney,East London,Intruder Alarm,1500.0,Cable Damage,Medium
8,Tesco Express Hackney,East London,Intruder Alarm,1500.0,Door Access Failure,High
9,Barclays Bank Stratford,East London,CCTV + Alarm,4500.0,Cable Damage,High


## Master Table

In [85]:
master = pd.read_sql('''
    SELECT 
        c.client_id,
        c.client_name,
        c.region,
        c.contract_type,
        c.monthly_fee_gbp,
        COUNT(DISTINCT i.job_id) as total_installs,
        SUM(DISTINCT i.install_cost_gbp) as total_install_revenue,
        COALESCE(co.total_callouts, 0) as total_callouts,
        COALESCE(co.total_sla_breaches, 0) as total_sla_breaches,
        COALESCE(co.total_repeat_faults, 0) as total_repeat_faults,
        co.avg_response_hrs
    FROM clients c
    LEFT JOIN installations i ON i.client_ref = c.client_id
    LEFT JOIN (
        SELECT 
            client_id,
            COUNT(*) as total_callouts,
            SUM(sla_breached) as total_sla_breaches,
            SUM(repeat_fault) as total_repeat_faults,
            ROUND(AVG(response_hrs), 1) as avg_response_hrs
        FROM callouts
        GROUP BY client_id
    ) co ON co.client_id = c.client_id
    GROUP BY c.client_id
''', conn)

In [86]:
total_mrr = master['monthly_fee_gbp'].sum()
total_arr = total_mrr*12

print(f'Total MRR: £{total_mrr:,.0f}')
print(f'Total ARR: £{total_arr:,.0f}')

Total MRR: £10,660
Total ARR: £127,920


In [87]:
master.groupby('region')['monthly_fee_gbp'].sum().sort_values(ascending=False)

region
East London     2890.0
North London    2200.0
Kent            1750.0
South London    1400.0
West London     1250.0
Essex           1170.0
Name: monthly_fee_gbp, dtype: float64

In [88]:
master.groupby('contract_type')['monthly_fee_gbp'].sum().sort_values(ascending=False)

contract_type
Maintenance        3840.0
Full Service       3720.0
Monitoring Only    3100.0
Name: monthly_fee_gbp, dtype: float64

In [89]:
total_callouts = master['total_callouts'].sum()
total_breaches = master['total_sla_breaches'].sum()

breach_rate = (total_breaches / total_callouts) * 100
print(f'Overall SLA breach rate: {breach_rate:.1f}%')

Overall SLA breach rate: 35.2%


In [91]:
regional_breach = master.groupby('region').agg(
    total_callouts=('total_callouts', 'sum'),
    total_breaches=('total_sla_breaches', 'sum')
)
regional_breach['breach_rate'] = (regional_breach['total_breaches'] / regional_breach['total_callouts'] * 100).round(1)
regional_breach.sort_values('breach_rate', ascending=False)

,total_callouts,total_breaches,breach_rate
region,,,
West London,13,6,46.2
South London,18,8,44.4
East London,44,17,38.6
Kent,20,7,35.0
Essex,18,6,33.3
North London,32,7,21.9


In [92]:
master['high_repeat_faults'] = master['total_repeat_faults'] >= 2

In [93]:
master['sla_breach_rate'] = master['total_sla_breaches'] / master['total_callouts'] * 100
master['high_sla_breach'] = master['sla_breach_rate'] > 40

In [94]:
master['at_risk'] = master['high_repeat_faults'] | master['high_sla_breach']

In [95]:
master[master['at_risk'] == True][['client_name', 'monthly_fee_gbp', 'total_repeat_faults', 'sla_breach_rate']]

,client_name,monthly_fee_gbp,total_repeat_faults,sla_breach_rate
0,Tesco Express Hackney,250.0,1,66.666667
3,DHL Warehouse Dagenham,350.0,1,50.000000
4,NHS Whipps Cross,150.0,1,57.142857
5,Lidl Walthamstow,1000.0,0,50.000000
6,Co-op Leyton,500.0,3,28.571429
7,Amazon Fulfilment Tilbury,350.0,0,60.000000
9,Canary Wharf Security Ltd,250.0,2,33.333333
10,McDonald's Romford,250.0,4,50.000000
11,Poundland Ilford,250.0,2,25.000000
12,Sports Direct Lakeside,750.0,1,50.000000


In [98]:
master.to_csv('master.csv', index=False)
print('Exported.')

Exported.
